# mini RLHF — walkthrough

Полный alignment-пайплайн на маленькой LM: **SFT → Reward Model (Bradley-Terry) → RL (PPO с KL-контролем)**.
Всё считается на CPU за секунды. Запускать из корня проекта `mini_rlhf`.

In [ ]:
import sys; sys.path.insert(0, '../src')
from mini_rlhf.config import RLHFConfig
from mini_rlhf.pipeline import run_pipeline

cfg = RLHFConfig.demo()
cfg.rl_algo = 'ppo'
res = run_pipeline(cfg, out_dir='../outputs/mini_rlhf')
res['metrics']

Ключевой результат: ground-truth reward почти не меняется после SFT (модель учит формат),
а после RL устойчиво растёт — выравнивание по reward-модели.

In [ ]:
for stage in ['base', 'sft', 'rl']:
    print(f'--- {stage} ---')
    for s in res['samples'][stage][:4]:
        print(' ', s)

In [ ]:
import matplotlib.pyplot as plt
h = res['rl_history']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h['gt_reward'], label='ground-truth reward'); ax[0].plot(h['rm_score'], label='RM score')
ax[0].set_title('RL: alignment progress'); ax[0].set_xlabel('iter'); ax[0].legend()
ax[1].plot(h['kl']); ax[1].set_title('KL(policy || reference)'); ax[1].set_xlabel('iter')
plt.tight_layout(); plt.show()

### Эксперимент: сила KL-штрафа
Слишком маленький `kl_coef` → reward hacking (повтор позитивных слов), большой → политика почти не меняется.

In [ ]:
for beta in [0.01, 0.05, 0.3]:
    cfg = RLHFConfig.demo(); cfg.kl_coef = beta
    r = run_pipeline(cfg, out_dir=f'../outputs/kl_{beta}')
    m = r['metrics']
    print(f"beta={beta}: RL reward={m['rl_gt_reward']:+.3f}, KL={m['final_kl_to_ref']:.2f}")